O objetivo deste notebook é unificar 6 arquivos CSV referentes à base de dados da CCEE, denominada "CONSUMO_HORARIO_PERFIL_AGENTE". Essa base contém informações sobre consumo bruto, ajustado e no ponto de conexão, por agente, ativo de carga, ramo de atividade e distribuidora conectada, considerando o período de comercialização no mês de referência. Para isso, foi utilizado o DuckDB para criar um Parquet, que permite trabalhar de forma eficiente com grandes volumes de dados diretamente do disco. Link dos dados abaixo

https://dadosabertos.ccee.org.br/dataset/consumo_horario_perfil_agente

Instalação da Biblioteca duckdb:

In [1]:
!pip install duckdb pandas openpyxl pyarrow

Procurando todos os arquivos CSV dentro de uma pasta específica e imprime os caminhos completos deles.

In [3]:
import duckdb
import os

pasta = r"C:\Users\Leticia\Documents\Izaque\Publicações linkedin\Cases\1. consumo_horario_perfil_agente"

arquivos = [os.path.join(pasta, f) for f in os.listdir(pasta) if f.endswith(".csv")]

print(arquivos)

['C:\\Users\\Leticia\\Documents\\Izaque\\Publicações linkedin\\Cases\\1. consumo_horario_perfil_agente\\consumo_horario_perfil_agente_202509.csv', 'C:\\Users\\Leticia\\Documents\\Izaque\\Publicações linkedin\\Cases\\1. consumo_horario_perfil_agente\\consumo_horario_perfil_agente_202510.csv', 'C:\\Users\\Leticia\\Documents\\Izaque\\Publicações linkedin\\Cases\\1. consumo_horario_perfil_agente\\consumo_horario_perfil_agente_202511.csv', 'C:\\Users\\Leticia\\Documents\\Izaque\\Publicações linkedin\\Cases\\1. consumo_horario_perfil_agente\\consumo_horario_perfil_agente_202512.csv', 'C:\\Users\\Leticia\\Documents\\Izaque\\Publicações linkedin\\Cases\\1. consumo_horario_perfil_agente\\consumo_horario_perfil_agente_202601.csv', 'C:\\Users\\Leticia\\Documents\\Izaque\\Publicações linkedin\\Cases\\1. consumo_horario_perfil_agente\\consumo_horario_perfil_agente_202602.csv']


Verificando o primeiro arquivo CSV da lista, lê ele com DuckDB e mostra as 5 primeiras linhas como um DataFrame

In [4]:
import duckdb

con = duckdb.connect()

arquivo = arquivos[0]

print(f"Lendo: {arquivo}")

con.execute(f"""
SELECT * FROM read_csv_auto('{arquivo}')
LIMIT 5
""").fetchdf()

Lendo: C:\Users\Leticia\Documents\Izaque\Publicações linkedin\Cases\1. consumo_horario_perfil_agente\consumo_horario_perfil_agente_202509.csv


,MES_REFERENCIA,DATA,PERIODO_COMERCIALIZACAO,CODIGO_CARGA,NOME_CARGA,DATA_MIGRACAO,CODIGO_PERFIL_AGENTE,SIGLA_PERFIL_AGENTE,CLASSE_PERFIL_AGENTE,NOME_EMPRESARIAL,...,CIDADE_CARGA,ESTADO_CARGA,SUBMERCADO,CODIGO_PERFIL_AGENTE_DISTRIBUIDORA,SIGLA_PERFIL_AGENTE_DISTRIBUIDORA,CAPACIDADE_CARGA,CONSUMO_CARGA_ACL,CONSUMO_CARGA_AJUSTADO_ACL,CONSUMO_CARGA_AJUSTADO,CONSUMO_CARGA_PONTO_CONEXAO
0,202509,2025-09-01,0,28,AES-SUL,2025-03-01,3,RGE SUL,Distribuidor,RGE SUL DISTRIBUIDORA DE ENERGIA S.A.,...,PORTO ALEGRE,RS,SUL,None,None,1540.25,479.685787,0,479.685787,468.263365
1,202509,2025-09-01,1,28,AES-SUL,2025-03-01,3,RGE SUL,Distribuidor,RGE SUL DISTRIBUIDORA DE ENERGIA S.A.,...,PORTO ALEGRE,RS,SUL,None,None,1540.25,431.065731,0,431.065731,420.477036
2,202509,2025-09-01,2,28,AES-SUL,2025-03-01,3,RGE SUL,Distribuidor,RGE SUL DISTRIBUIDORA DE ENERGIA S.A.,...,PORTO ALEGRE,RS,SUL,None,None,1540.25,404.947344,0,404.947344,394.776462
3,202509,2025-09-01,3,28,AES-SUL,2025-03-01,3,RGE SUL,Distribuidor,RGE SUL DISTRIBUIDORA DE ENERGIA S.A.,...,PORTO ALEGRE,RS,SUL,None,None,1540.25,393.093319,0,393.093319,382.891059
4,202509,2025-09-01,4,28,AES-SUL,2025-03-01,3,RGE SUL,Distribuidor,RGE SUL DISTRIBUIDORA DE ENERGIA S.A.,...,PORTO ALEGRE,RS,SUL,None,None,1540.25,398.214449,0,398.214449,388.079058


In [ ]:
Verificando quantas linhas existem no arquivo CSV visto acima

In [5]:
con.execute(f"""
SELECT COUNT(*) 
FROM read_csv_auto('{arquivo}')
""").fetchall()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[(32271120,)]

Criando uma tabela DuckDB (base_unificada) juntando todos os CSVs da pasta.

In [7]:
con.execute(f"""
CREATE TABLE base_unificada AS
SELECT * 
FROM read_csv_auto(
    '{pasta}/*.csv',
    delim=';',
    quote='"',
    escape='"'
)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
Consultando na tabela criada (base_unificada) o total de registros existente.

In [8]:
con.execute("SELECT COUNT(*) FROM base_unificada").fetchall()

[(195448968,)]

In [ ]:
verificando nomes de colunas, tipos de dados da tabela criada

In [9]:
con.execute("DESCRIBE base_unificada").fetchdf()

,column_name,column_type,null,key,default,extra
0,MES_REFERENCIA,BIGINT,YES,None,None,None
1,DATA,DATE,YES,None,None,None
2,PERIODO_COMERCIALIZACAO,BIGINT,YES,None,None,None
3,CODIGO_CARGA,BIGINT,YES,None,None,None
4,NOME_CARGA,VARCHAR,YES,None,None,None
5,DATA_MIGRACAO,DATE,YES,None,None,None
6,CODIGO_PERFIL_AGENTE,BIGINT,YES,None,None,None
7,SIGLA_PERFIL_AGENTE,VARCHAR,YES,None,None,None
8,CLASSE_PERFIL_AGENTE,VARCHAR,YES,None,None,None
9,NOME_EMPRESARIAL,VARCHAR,YES,None,None,None


In [ ]:
Exportando a tabela criada em um arquivo parquet.

In [10]:
con.execute("""
COPY base_unificada 
TO 'base_final.parquet' 
(FORMAT 'parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Verificando o arquivo parquet criado, consultado o número de registro existente. Observe que é o mesmo valor encontrado acima, ao unificar os arquivos csv.

In [12]:
con.execute("""
SELECT COUNT(*) 
FROM 'base_final.parquet'
""").fetchall()

[(195448968,)]

A geração do Parquet foi fundamental para transformar um conjunto de dados grande e difícil de manipular em uma base leve, rápida e eficiente. Ao reduzir drasticamente o tamanho dos arquivos e permitir leituras otimizadas, o DuckDB consegue consultar apenas o necessário, tornando análises sobre milhões de registros muito mais rápidas. Na prática, isso elevou seu processo de um tratamento bruto de dados para um pipeline otimizado e pronto para uso real.